In [1]:
import torch
from usta_model import UstaModel
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of united"

tokens = u_tokenizer.encode(prompt)
tokens

tensor([ 0, 61,  1, 61,  2, 61,  3])

In [2]:
torch.manual_seed(1)
u_model = UstaModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=4,context_length=32)

sentence_meanings_with_attention_context = u_model(tokens)
sentence_meanings_with_attention_context.shape

torch.Size([7, 4])

In [3]:
out = u_model(tokens)
out

tensor([[-0.1370, -0.3226,  0.1265,  0.1362],
        [-0.4744, -0.2934, -0.0552,  0.0922],
        [-0.5055, -0.2895, -0.0209,  0.1401],
        [-0.4333, -0.2986, -0.0499,  0.0810],
        [-0.2547, -0.2411,  0.1826,  0.2178],
        [-0.1604, -0.3156,  0.1119,  0.1300],
        [-0.4393, -0.2969, -0.0482,  0.0849]], grad_fn=<AddmmBackward0>)

In [4]:
u_model

UstaModel(
  (embedding): Embedding(64, 4)
  (pos_embedding): Embedding(32, 4)
  (self_attention): UstaMultiHeadAttention(
    (multi_head_attention): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
    )
    (projection): Linear(in_features=4, out_features=4, bias=True)
  )
  (norm): UstaLayerNorm()
  (mlp): UstaMLP(
    (gate_proj): Linear(in_features=4, out_features=4, bias=True)
    (up_proj): Linear(in_features=4, out_features=4, bias=True)
    (down_proj): Linear(in_features=4, out_features=4, bias=True)
    (gelu): GELU()
  )
)

In [5]:
from usta_layer_norm import UstaLayerNorm

norm_layer = UstaLayerNorm(4)
norm_layer(out)

tensor([[ 0.9774, -1.3497, -0.5733,  0.9456],
        [-0.0395, -0.9133, -0.6878,  1.6405],
        [ 1.6875, -0.7836, -0.1973, -0.7067],
        [-0.8302, -0.0373, -0.7773,  1.6448],
        [-1.0719,  0.9883, -0.9254,  1.0090],
        [ 1.5746, -1.1255,  0.0519, -0.5010],
        [ 0.0441,  0.7421, -1.6458,  0.8596]], grad_fn=<MulBackward0>)

![transformers architecture](https://heidloff.net/assets/img/2023/02/transformers.png)

In [6]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

q_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
q_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

/Users/iremdinc/.pyenv/versions/3.13.3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1090.18it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [7]:
q_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

![Gelu](https://miro.medium.com/v2/resize:fit:1400/1*O5E-huBuY1UTHMmM--rhLQ.png)

In [8]:
import torch.nn as nn

class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (
      1 + torch.tanh(
          torch.sqrt(torch.tensor(2 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
        )
    )


gelu = GELU()

example_tensor = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
gelu(example_tensor)

tensor([[0.8412, 1.9546, 2.9964],
        [3.9999, 5.0000, 6.0000]])

In [9]:
# import torch funtions 

import torch.nn.functional as F

F.gelu(example_tensor, approximate="tanh")

tensor([[0.8412, 1.9546, 2.9964],
        [3.9999, 5.0000, 6.0000]])